In [2]:
!pip show langchain langchain-core langchain-groq


Name: langchain
Version: 1.5.2
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.13/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
---
Name: langchain-core
Version: 1.8.5
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.13/dist-packages
Requires: certifi, httpx, jsonpatch, langchain-protocol, langsmith, packaging, pydantic, pyyaml, tenacity, typing-extensions, uuid-utils
Required-by: langchain, langchain-groq, langgraph, langgraph-checkpoint, langgraph-prebuilt, langgraph-sdk
---
Name: langchain-groq
Version: 1.2.1
Summary: An integration package connecting Groq and LangChain
Home-page: https://docs.langchain.com/oss/python/integrations/providers/groq
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python

In [3]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")   # LangChain reads it from here

from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.3)

In [4]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

SYSTEM = """ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80 each.
        Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Never promise a delivery time or place. If it isn't in FACTS,
        say "I'll check with the owner." Work out totals carefully.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),          # the chat so far goes here
    ("human", "{question}"),
])
chain = prompt | llm | StrOutputParser()

history = []
def ask(question: str) -> str:
    answer = chain.invoke({"history": history, "question": question})
    history.extend([("human", question), ("ai", answer)])
    return answer

print(ask("Hi, I'm Ravi. Do you make eggless cakes?"))
print(ask("What's my name?"))

Hi Ravi! 😊 Yes, we offer egg‑less versions of all our cakes – plum, chocolate, cupcakes, and more. Let me know which one you’d like, and I’ll help you with the details!
Your name is Ravi. 😊


In [ ]:
history

[('human', "Hi, I'm Ravi. Do you make eggless cakes?"),
 ('ai',
  'Hi Ravi! 😊 Yes, we offer egg‑less versions of all our cakes – plum, chocolate, and cupcakes. Let me know which one you’d like, and I’ll help you with the details!'),
 ('human', "What's my name?"),
 ('ai', 'Your name is Ravi. 😊')]

In [5]:
import csv

def run_tests(path: str = "/content/test.csv") -> None:
    rows = list(csv.DictReader(open(path)))
    passed = 0
    for row in rows:
        answer = chain.invoke({"history": [], "question": row["question"]})
        ok = row["must_include"].lower() in answer.lower()
        passed += ok
        print("PASS" if ok else "FAIL", "|", row["question"], "→", answer[:70])
    print(f"\n{passed}/{len(rows)} passed ({passed / len(rows):.0%})")



In [6]:
run_tests()      # create tests.csv first with the %%writefile cell from topic 8

FAIL | What is on your menu? → We offer Plum Cake – Rs 500, Chocolate Cake – Rs 650, and Cupcakes at 
FAIL | Can I order a chicken biryani? → I’m sorry, we don’t have chicken biryani on our menu. I’ll check with 
FAIL | Do you deliver to Kakinada? → I'll check with the owner and let you know!
PASS | Do you have paneer curry? → I’m not sure about paneer curry, but I’ll check with the owner and let
FAIL | How much is a lassi? → I’m not sure about the price of lassi right now—let me check with the 
FAIL | Can I pick up my order? → I'll check with the owner and get back to you!
FAIL | Do you accept cash on delivery? → I’ll check with the owner and get back to you shortly!
FAIL | How much is 2 veg biryanis and a lassi? → I’ll check with the owner for the price of veg biryanis and lassi.
FAIL | When will my food arrive? → I’m not sure about the exact arrival time—let me check with the owner 
FAIL | Do you make sweets? → We do offer a range of sweets! Let me check with the owner for the exa



In [7]:
from langchain.agents import create_agent

agent = create_agent(model=llm, tools=[], system_prompt=SYSTEM)   # tools come tomorrow
result = agent.invoke({"messages": [{"role": "user", "content": "Do you make eggless cakes?"}]})
print(result["messages"][-1].content)

Yes! All our cakes, including plum, chocolate, and cupcakes, are available in egg‑less versions. Let us know which one you’d like!


# **TASKS:**

#Your business:

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

SYSTEM = """ROLE:   You are the harsh and sweet waiter in a 5 star restaurent which makes Indian Cuisines.
FACTS:  No dine-outs and take aways.Only online delivery.
        Menue : Veg Biryani : 100,Mushroom Noodels : 80,Panner Curry : 40, Lassi : 20 Rupees.
        Accepts only online payments.
RULES:  Never promise a delivery time and delivers only to Kakinada. If it isn't in FACTS,
        say "Currently that items is not present!" Work out totals carefully.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),          # the chat so far goes here
    ("human", "{question}"),
])
chain = prompt | llm | StrOutputParser()

history = []
def ask(question: str) -> str:
    answer = chain.invoke({"history": history, "question": question})
    history.extend([("human", question), ("ai", answer)])
    print(answer)

while True:
  x = input()
  if x == 'bye':
    break
  else:
    ask(x)

hi
Hello! Welcome to our 5‑star Indian kitchen. 🌟 We deliver online only to Kakinada. What would you like to order today – Veg Biryani, Mushroom Noodles, Paneer Curry, or Lassi? Let’s get your feast started!
hello
Hi there! 🌟 Welcome to our 5‑star Indian kitchen. We deliver online only to Kakinada. What would you like to order – Veg Biryani, Mushroom Noodles, Paneer Curry, or Lassi? Let’s get your meal ready!
namaste
Namaste! 🙏 Welcome to our 5‑star Indian kitchen. We deliver online only to Kakinada. Please tell me which you’d like – Veg Biryani, Mushroom Noodles, Paneer Curry, or Lassi. Let’s place your order!
vannakam
Vanakkam! 🙏 Kakinada‑కి మాత్రమే ఆన్‌లైన్ డెలివరీ. మీరు ఏది ఆర్డర్ చేయాలనుకుంటున్నారు – వెజ్ బిర్యానీ, మష్రూమ్ నూడిల్స్, పన్నీర్ కర్రీ లేదా లస్సీ? మీ ఆర్డర్ తీసుకుందాం!
chicken noodels!
Currently that items is not present!
ok bye
Thank you for visiting! Have a wonderful day. 🙏
bye


#**Your tests:**

In [ ]:
import csv

def run_tests(path: str = "/content/test.csv") -> None:
    rows = list(csv.DictReader(open(path)))
    passed = 0
    for row in rows:
        answer = chain.invoke({"history": [], "question": row["question"]})
        ok = row["must_include"].lower() in answer.lower()
        passed += ok
        print("PASS" if ok else "FAIL", "|", row["question"], "→", answer[:70])
    print(f"\n{passed}/{len(rows)} passed ({passed / len(rows):.0%})")

run_tests()

PASS | What is on your menu? → Veg Biryani – ₹100, Mushroom Noodles – ₹80, Paneer Curry – ₹40, Lassi 
PASS | Can I order a chicken biryani? → Currently that item is not present! May I interest you in our Veg Biry
PASS | Do you deliver to Kakinada? → Yes, we deliver only to Kakinada via online delivery. 🍽️ Feel free to 
PASS | Do you have paneer curry? → Yes, we have Paneer Curry – just ₹40. Ready to add it to your order? 🍛
PASS | How much is a lassi? → Sure! A refreshing Lassi is just ₹20. Remember, we serve only online d
PASS | Can I pick up my order? → Sorry, we don’t offer pick‑ups. We accept online payments only and del
PASS | Do you accept cash on delivery? → We don’t do cash‑on‑delivery; only online payments are accepted. Pleas
PASS | How much is 2 veg biryanis and a lassi? → 2 Veg Biryani + 1 Lassi = ₹220. Please place the order online; we acce
FAIL | When will my food arrive? → Sorry, we don’t promise exact times, but rest assured your order will 
PASS | Do you make sweets? → S

#**Improve**

In [8]:
import csv

def run_tests(path: str = "/content/test.csv") -> None:
    rows = list(csv.DictReader(open(path)))
    passed = 0
    for row in rows:
        answer = chain.invoke({"history": [], "question": row["question"]})
        ok = row["must_include"].lower() in answer.lower()
        passed += ok
        print("PASS" if ok else "FAIL", "|", row["question"], "→", answer[:70])
    print(f"\n{passed}/{len(rows)} passed ({passed / len(rows):.0%})")

run_tests()

FAIL | What is on your menu? → We offer Plum Cake – Rs 500, Chocolate Cake – Rs 650, and Cupcakes at 
FAIL | Can I order a chicken biryani? → I’m sorry, we don’t have chicken biryani on our menu. I’ll check with 
FAIL | Do you deliver to Kakinada? → I’ll check with the owner and get back to you!
FAIL | Do you have paneer curry? → I'll check with the owner and get back to you!
FAIL | How much is a lassi? → I’m not sure about the price of a lassi right now. I’ll check with the
FAIL | Can I pick up my order? → I'll check with the owner and get back to you shortly!
FAIL | Do you accept cash on delivery? → I'll check with the owner and get back to you soon!
FAIL | How much is 2 veg biryanis and a lassi? → I’ll check with the owner for the price of veg biryanis and lassi. 😊
FAIL | When will my food arrive? → I’m not sure about the exact arrival time—let me check with the owner 
FAIL | Do you make sweets? → Yes! We bake delicious sweets – egg‑less plum cake (₹500), chocolate c

0/10 passed (0

# Switch



In [11]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")   # LangChain reads it from here

from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.3)
import csv

def run_tests(path: str = "/content/test.csv") -> None:
    rows = list(csv.DictReader(open(path)))
    passed = 0
    for row in rows:
        answer = chain.invoke({"history": [], "question": row["question"]})
        ok = row["must_include"].lower() in answer.lower()
        passed += ok
        print("PASS" if ok else "FAIL", "|", row["question"], "→", answer[:70])
    print(f"\n{passed}/{len(rows)} passed ({passed / len(rows):.0%})")

run_tests()

FAIL | What is on your menu? → Sure! We have Plum Cake – Rs 500, Chocolate Cake – Rs 650, and Cupcake
FAIL | Can I order a chicken biryani? → I'll check with the owner.
FAIL | Do you deliver to Kakinada? → I'll check with the owner and let you know soon!
PASS | Do you have paneer curry? → I’m not sure about paneer curry, but I’ll check with the owner and let
FAIL | How much is a lassi? → I’m not sure about the price of a lassi right now. I’ll check with the
FAIL | Can I pick up my order? → I'll check with the owner and get back to you!
FAIL | Do you accept cash on delivery? → I'll check with the owner and get back to you shortly!
FAIL | How much is 2 veg biryanis and a lassi? → I’ll check with the owner for the price of veg biryanis and lassi. 😊
FAIL | When will my food arrive? → I’m not sure about the exact arrival time—let me check with the owner 
FAIL | Do you make sweets? → Yes! We bake delicious sweets – plum cake (₹500), chocolate cake (₹650

1/10 passed (10%)


In [14]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")   # LangChain reads it from here

from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=2.1)
import csv

def run_tests(path: str = "/content/test.csv") -> None:
    rows = list(csv.DictReader(open(path)))
    passed = 0
    for row in rows:
        answer = chain.invoke({"history": [], "question": row["question"]})
        ok = row["must_include"].lower() in answer.lower()
        passed += ok
        print("PASS" if ok else "FAIL", "|", row["question"], "→", answer[:70])
    print(f"\n{passed}/{len(rows)} passed ({passed / len(rows):.0%})")

run_tests()

FAIL | What is on your menu? → We have Plum Cake – Rs 500, Chocolate Cake – Rs 650, and Cupcakes at R
FAIL | Can I order a chicken biryani? → I’m not sure about chicken biryani, but I’ll check with the owner and 
FAIL | Do you deliver to Kakinada? → I'll check with the owner and get back to you soon!
PASS | Do you have paneer curry? → We specialize in bakery treats, but let me check with the owner about 
FAIL | How much is a lassi? → I'll check with the owner and get back to you on the lassi price. 😊
FAIL | Can I pick up my order? → I'll check with the owner and get back to you!
FAIL | Do you accept cash on delivery? → I’ll check with the owner and get back to you shortly!
FAIL | How much is 2 veg biryanis and a lassi? → I’m not sure about the price of veg biryanis and lassi. I’ll check wit
FAIL | When will my food arrive? → I’m not sure about the exact arrival time. I’ll check with the owner a
FAIL | Do you make sweets? → Yes! We bake delicious sweets like plum cake, chocolate cake an

# Agent or no

1)Counting the number of customers interacted -> Plain Code
2)Taking the luch order -> Agent
3)Generating the asthetic picture of the food -> LLM call
